# ASR Metrics Evaluation Showcase
Ten notatnik demonstruje wykorzystanie zoptymalizowanego modułu `metrics.py` do kompleksowej oceny systemu ASR.

In [ ]:
!pip install -r requirements.txt

## 1. Import i przygotowanie danych
Generujemy przykładowe dane wejściowe dla metryk. W prawdziwym scenariuszu użyłbyś wyników ze swojego systemu ASR.

In [ ]:
import numpy as np
import soundfile as sf
import time
from metrics import eval_all_metrics, preload_models

# Generowanie plików testowych audio (1-sekundowy sygnał sinus)
sample_rate = 16000
t = np.linspace(0, 1.0, sample_rate)
sf.write('test_ref.flac', np.sin(2 * np.pi * 440. * t), sample_rate)
sf.write('test_hyp.flac', np.sin(2 * np.pi * 440. * t + 0.1), sample_rate)

ytrue_batch = ["the quick brown fox jumps over the lazy dog"]
ypred_batch = ["the quick brown fox jumps over the dog"]
ref_audios = ['test_ref.flac']
hyp_audios = ['test_hyp.flac']
decode_times = [0.5]  # Przykładowy czas trwania inferencji Twojego modelu ASR
encode_times = [0.2]

## 2. Pre-loading modeli (SpeechBrain & SeMaScore)
Aby pominąć czas ładowania wag z dysku do GPU podczas samej inferencji, ładujemy je z wyprzedzeniem.

In [ ]:
print("Ładowanie modeli do pamięci...")
preload_models()

## 3. Wywołanie ewaluacji
Uruchamiamy główną metodę ewaluacji i mierzymy czysty czas wyliczania metryk w oparciu o batch.

In [ ]:
print("Trwa ewaluacja batcha...")
start_t = time.time()

agg_res, ind_res = eval_all_metrics(
    ytrue_texts=ytrue_batch,
    ypred_texts=ypred_batch,
    ytrue_audio_paths=ref_audios,
    ypred_audio_paths=hyp_audios,
    decode_times=decode_times,
    encode_times=encode_times,
    log=False
)

eval_duration = time.time() - start_t

print('\n' + '='*50)
print('WYNIKI EWALUACJI (AGGREGATE)')
print('='*50)
for metric_name, value in agg_res.items():
    if isinstance(value, float):
        print(f'{metric_name.upper():<20}: {value:.4f}')
    else:
        print(f'{metric_name.upper():<20}: {value}')
print('='*50)
print(f'Czas wyliczania wszystkich metryk (inferencja): {eval_duration:.4f} sekund')
print('='*50)